#######################################
Here we go!
#######################################
Introducing the concept of Microbiota similarity score (MSS)
In this concept, we intend to ask, after dysbiosis or even after an intervention to correct dysbiosis, how do we know how far is our target gut microbiota towards the goal, the goal of returning to a normal gut profile? Can we have a machine learning (ML) way of doing this, of exploring the profiles before and after and then evaluate how far it is to reach our goal and how is the intervention doing? If this can be achieved, then we found a way to apply ML beyond the usually classification and biomarker identification.

In [ ]:
# Install required packages
!pip install shap scikit-learn matplotlib seaborn openpyxl pandas numpy scipy -q

In [ ]:
# Setting the ground and leveling it for the upcoming game
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import mannwhitneyu, pearsonr

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, cross_val_predict, GridSearchCV, LeaveOneOut
)
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    accuracy_score, f1_score, recall_score, precision_score
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils import resample

matplotlib.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white'
})

SET1 = plt.cm.get_cmap('Set1').colors
print('Ground is ready, all libraries are loaded successfully.')

In [ ]:
# ── UPLOAD FILES IN COLAB ──────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()

# ── DATA LOADING ──────────────────────────────────────────────────────────────
TRAIN_FILE = 'Dataset2.xlsx'   # Change path if needed
TEST_FILE  = 'Dataset-1.xlsx'  # Change path if needed

df_train = pd.read_excel(TRAIN_FILE)
df_test  = pd.read_excel(TEST_FILE)

print(f'Training set shape : {df_train.shape}')
print(f'Training classes   : {df_train["Status"].value_counts().to_dict()}')
print(f'Test set shape     : {df_test.shape}')
print(f'Test timepoints    : {df_test["Status"].value_counts().to_dict()}')

In [ ]:
# ── FEATURE FILTRATION: This is intended to retain taxa detected in ≥5% of samples ────
feature_cols = [c for c in df_train.columns if c not in ('Genus', 'Status')]

prevalence = (df_train[feature_cols] > 0).mean()
kept_features = prevalence[prevalence >= 0.05].index.tolist()

print(f'Features before filtration : {len(feature_cols)}')
print(f'Features after ≥5% filter  : {len(kept_features)}')

X_all = df_train[kept_features].values
y_raw = df_train['Status'].values
le    = LabelEncoder()          # CRC=0, NC=1
y_all = le.fit_transform(y_raw)
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [ ]:
# Differential Feature Selection for Top 20 genera : Wilcoxon-Mann-Whitney (WMW)
# ── WMW test on all kept features ─────────────────────────────────────────────
df_feat = df_train[kept_features + ['Status']].copy()
crc_idx = df_train['Status'] == 'CRC'
nc_idx  = df_train['Status'] == 'NC'

wmw_results = []
for feat in kept_features:
    crc_vals = df_feat.loc[crc_idx, feat].values
    nc_vals  = df_feat.loc[nc_idx,  feat].values
    stat, pval = mannwhitneyu(crc_vals, nc_vals, alternative='two-sided')
    wmw_results.append({'feature': feat, 'statistic': stat, 'pvalue': pval})

wmw_df = pd.DataFrame(wmw_results).sort_values('statistic', ascending=False)

# Select top 20 by absolute effect size (WMW statistic) with p < 0.05
wmw_sig = wmw_df[wmw_df['pvalue'] < 0.05]
top20   = wmw_sig.head(20)['feature'].tolist()

# Fallback: if < 20 significant, take top 20 overall
if len(top20) < 20:
    top20 = wmw_df.head(20)['feature'].tolist()

print(f'Top 20 differential features (WMW):')
for i, f in enumerate(top20, 1):
    row = wmw_df[wmw_df['feature'] == f].iloc[0]
    print(f'  {i:2d}. {f:30s}  stat={row["statistic"]:.0f}  p={row["pvalue"]:.4e}')

In [ ]:
# Use only top-20 features for downstream modeling
X = df_train[top20].values
y = y_all
print(f'Modeling feature matrix: {X.shape}')

In [ ]:
# Classifier Training: 10-Fold CV + Nested Hyperparameter Tuning
# ── Model definitions with hyperparameter grids ────────────────────────────────
scaler = StandardScaler()

models = {
    'Logistic Regression': {
        'pipeline': Pipeline([('scaler', StandardScaler()),
                              ('clf', LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42))]),
        'param_grid': {'clf__C': [0.001, 0.01, 0.1, 1, 10, 100]}
    },
    'LASSO Regression': {
        'pipeline': Pipeline([('scaler', StandardScaler()),
                              ('clf', LogisticRegression(penalty='l1', solver='liblinear', max_iter=2000, random_state=42))]),
        'param_grid': {'clf__C': [0.001, 0.01, 0.1, 1, 10, 100]}
    },
    'SVM': {
        'pipeline': Pipeline([('scaler', StandardScaler()),
                              ('clf', SVC(probability=True, random_state=42))]),
        'param_grid': {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 'auto']}
    },
    'Random Forest': {
        'pipeline': Pipeline([('scaler', StandardScaler()),
                              ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))]),
        'param_grid': {'clf__max_depth': [None, 5, 10], 'clf__min_samples_split': [2, 5]}
    }
}

print('Models defined:', list(models.keys()))

In [ ]:
# ── Nested CV: outer 10-fold, inner 5-fold for hyperparam tuning ──────────────
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=5,  shuffle=True, random_state=42)

results = {}

for name, cfg in models.items():
    print(f'Training: {name} ...', end=' ')
    gs = GridSearchCV(cfg['pipeline'], cfg['param_grid'],
                      cv=inner_cv, scoring='roc_auc', n_jobs=-1, refit=True)

    all_probs  = np.zeros(len(y))
    all_preds  = np.zeros(len(y), dtype=int)

    for train_idx, test_idx in outer_cv.split(X, y):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        gs.fit(X_tr, y_tr)
        proba = gs.predict_proba(X_te)[:, 1]   # prob of CRC=0 → NC=1
        all_probs[test_idx] = proba
        all_preds[test_idx] = gs.predict(X_te)

    auc  = roc_auc_score(y, all_probs)
    acc  = accuracy_score(y, all_preds)
    f1   = f1_score(y, all_preds, average='macro')
    sens = recall_score(y, all_preds, pos_label=0)        # CRC sensitivity
    spec = recall_score(y, all_preds, pos_label=1)        # NC specificity
    cm   = confusion_matrix(y, all_preds)
    fpr, tpr, _ = roc_curve(y, all_probs)

    # Refit best model on full data for SHAP / MSS
    gs.fit(X, y)

    results[name] = {
        'auc': auc, 'accuracy': acc, 'f1': f1,
        'sensitivity': sens, 'specificity': spec,
        'cm': cm, 'fpr': fpr, 'tpr': tpr,
        'probs': all_probs, 'preds': all_preds,
        'fitted_model': gs
    }
    print(f'AUC = {auc:.4f}')

print('\nAll models trained.')

In [ ]:
# Performance Metrics Table

# ── Build metrics table ──────────────────
metrics_data = []
for name, res in results.items():
    metrics_data.append({
        'Model': name,
        'AUC':         f"{res['auc']:.4f}",
        'Accuracy':    f"{res['accuracy']:.4f}",
        'Sensitivity': f"{res['sensitivity']:.4f}",
        'Specificity': f"{res['specificity']:.4f}",
        'F1 Score':    f"{res['f1']:.4f}"
    })

metrics_df = pd.DataFrame(metrics_data)

# Identify best model
best_model_name = max(results, key=lambda k: results[k]['auc'])
print(f'Best model by AUC: {best_model_name}  (AUC = {results[best_model_name]["auc"]:.4f})')
print()
print(metrics_df.to_string(index=False))

# Let's have quality figures now
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.axis('off')

col_labels = metrics_df.columns.tolist()
cell_text  = metrics_df.values.tolist()

table = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# Header styling
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Row styling
for i in range(1, len(cell_text) + 1):
    row_model = cell_text[i - 1][0]
    bg = '#d5e8d4' if row_model == best_model_name else ('#f2f2f2' if i % 2 == 0 else 'white')
    for j in range(len(col_labels)):
        table[i, j].set_facecolor(bg)
        if row_model == best_model_name:
            table[i, j].set_text_props(fontweight='bold')

plt.title('Model Performance Metrics (10-Fold Cross-Validation, Macro-Averaged)\n'
          f'★ Selected Model: {best_model_name}',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('metrics_table.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ROC-AUC Curves for All Models

fig, ax = plt.subplots(figsize=(8, 7))

colors  = [SET1[i] for i in range(len(results))]
lwidths = [3 if k == best_model_name else 1.8 for k in results]
lstyles = ['-' if k == best_model_name else '--' for k in results]

for (name, res), col, lw, ls in zip(results.items(), colors, lwidths, lstyles):
    label = f"{name} (AUC = {res['auc']:.4f})"
    if name == best_model_name:
        label = f"★ {label}"
    ax.plot(res['fpr'], res['tpr'], color=col, lw=lw, ls=ls, label=label)

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random Chance')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate (1 − Specificity)', fontsize=13)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=13)
ax.set_title('Receiver Operating Characteristic (ROC) Curves\nCRC vs. Normal Control (NC) — 10-Fold Cross-Validation',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', frameon=True, framealpha=0.9, fontsize=10)
ax.fill_between(results[best_model_name]['fpr'],
                results[best_model_name]['tpr'],
                alpha=0.08, color=colors[list(results.keys()).index(best_model_name)])

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# Let's also get those Confusion Matrices (All Models)

n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
if n_models == 1:
    axes = [axes]

class_labels = ['CRC', 'NC']

for ax, (name, res) in zip(axes, results.items()):
    cm_norm = res['cm'].astype(float) / res['cm'].sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=res['cm'], fmt='d',
        cmap='Blues', ax=ax,
        xticklabels=class_labels, yticklabels=class_labels,
        linewidths=0.5, linecolor='gray',
        cbar_kws={'shrink': 0.8},
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    star = '★ ' if name == best_model_name else ''
    ax.set_title(f'{star}{name}\nAUC = {res["auc"]:.4f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)

plt.suptitle('Confusion Matrices — CRC vs. NC (10-Fold CV)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# SHAP Beeswarm Summary Plot (Best Model)
print(f'Computing SHAP values for best model: {best_model_name}')
best_gs    = results[best_model_name]['fitted_model']
best_clf   = best_gs.best_estimator_

# Transform X with the pipeline's scaler steps (all but last)
from sklearn.pipeline import Pipeline as SKPipeline
def get_transformed_X(pipeline, X):
    Xt = X.copy()
    for name_step, step in pipeline.steps[:-1]:
        Xt = step.transform(Xt)
    return Xt

X_scaled = get_transformed_X(best_clf, X)
clf_step  = best_clf.named_steps['clf']

# Use TreeExplainer for RF, LinearExplainer for linear models, KernelExplainer for SVM
if best_model_name == 'Random Forest':
    explainer   = shap.TreeExplainer(clf_step)
    shap_values = explainer.shap_values(X_scaled)
    # For binary classification, shap_values is a list; take class 0 (CRC)
    sv = shap_values[0] if isinstance(shap_values, list) else shap_values
elif best_model_name in ('Logistic Regression', 'LASSO Regression'):
    explainer   = shap.LinearExplainer(clf_step, X_scaled)
    sv          = explainer.shap_values(X_scaled)
    if isinstance(sv, list):
        sv = sv[0]
else:  # SVM
    # Sample 100 background points for speed
    bg   = shap.sample(X_scaled, min(100, len(X_scaled)), random_state=42)
    explainer = shap.KernelExplainer(clf_step.predict_proba, bg)
    sv        = explainer.shap_values(X_scaled[:100], nsamples=100)
    if isinstance(sv, list):
        sv = sv[0]
    X_scaled  = X_scaled[:100]   # match rows for plotting

# Beeswarm plot
shap_df = pd.DataFrame(sv, columns=top20)

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    sv, X_scaled,
    feature_names=top20,
    plot_type='dot',
    max_display=20,
    show=False,
    color_bar=True
)
plt.title(f'SHAP Beeswarm Summary — {best_model_name}\nGlobal Feature Importance (CRC vs. NC)',
          fontsize=13, fontweight='bold')
plt.xlabel('SHAP Value (Impact on Model Output)', fontsize=12)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# It is the moment for Microbiota Similarity Score (MSS) — Calibrated by Brier Score

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# ── Calibrate best model with Platt scaling ─────────────
best_raw_clf = best_gs.best_estimator_  # already fitted pipeline

# Re-extract base estimator for calibration
calibrated_pipeline = CalibratedClassifierCV(
    best_raw_clf, method='sigmoid', cv=5
)
calibrated_pipeline.fit(X, y)

# Brier score before/after calibration
probs_raw  = results[best_model_name]['probs']
probs_cal  = calibrated_pipeline.predict_proba(X)[:, 1]

brier_raw = brier_score_loss(y, probs_raw)
brier_cal = brier_score_loss(y, probs_cal)

print(f'Brier Score — Uncalibrated : {brier_raw:.4f}')
print(f'Brier Score — Calibrated   : {brier_cal:.4f}')
print(f'(Lower is better; < 0.25 = acceptable calibration)')

# ── Calibration curve ─────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

frac_pos_raw, mean_pred_raw = calibration_curve(y, probs_raw, n_bins=10)
frac_pos_cal, mean_pred_cal = calibration_curve(y, probs_cal, n_bins=10)

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Perfect Calibration')
ax.plot(mean_pred_raw, frac_pos_raw, 's-', color=SET1[0], lw=2,
        label=f'Uncalibrated (Brier={brier_raw:.4f})')
ax.plot(mean_pred_cal, frac_pos_cal, 'o-', color=SET1[1], lw=2,
        label=f'Calibrated (Brier={brier_cal:.4f})')

ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Fraction of Positives', fontsize=12)
ax.set_title('Calibration Curve — Microbiota Similarity Score (MSS)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, frameon=True)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# MSS on Dataset-1 (in-house dataset with T1/T2/T3) — Gut Microbiota Similarity to CRC vs NC

# ── Align Dataset-1 features with top20 ───────────────────────────────────────

# Ensure same feature set; fill missing with 0

missing_in_test = [f for f in top20 if f not in df_test.columns]
for mf in missing_in_test:
    df_test[mf] = 0.0

X_new       = df_test[top20].values
timepoints  = df_test['Status'].values   # T1, T2, T3
sample_ids  = df_test['Genus'].values

# ── Compute MSS (probability of being NC) ────────────────────────────────────
# We define MSS as P(NC) — higher MSS = more similar to NC profile
# le.classes_: CRC=0, NC=1  →  predict_proba[:,1] = P(NC)
proba_new = calibrated_pipeline.predict_proba(X_new)
mss_nc    = proba_new[:, 1]   # similarity to NC
mss_crc   = proba_new[:, 0]   # similarity to CRC

df_mss = pd.DataFrame({
    'SampleID':    sample_ids,
    'Timepoint':   timepoints,
    'MSS_NC':      mss_nc,
    'MSS_CRC':     mss_crc
})

# Summary statistics
print('MSS Summary by Timepoint (NC Similarity Score):')
print(df_mss.groupby('Timepoint')[['MSS_NC', 'MSS_CRC']].describe().round(4))

In [ ]:
# ── Statistical comparisons between timepoints ────────────────────────────────
t1_mss = df_mss.loc[df_mss['Timepoint'] == 'T1', 'MSS_NC'].values
t2_mss = df_mss.loc[df_mss['Timepoint'] == 'T2', 'MSS_NC'].values
t3_mss = df_mss.loc[df_mss['Timepoint'] == 'T3', 'MSS_NC'].values

t12_stat, t12_p = stats.ttest_ind(t1_mss, t2_mss, equal_var=False)
t13_stat, t13_p = stats.ttest_ind(t1_mss, t3_mss, equal_var=False)
t23_stat, t23_p = stats.ttest_ind(t2_mss, t3_mss, equal_var=False)

def sig_label(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

print(f'Welch t-test T1 vs T2: t={t12_stat:.3f}, p={t12_p:.4f}  {sig_label(t12_p)}')
print(f'Welch t-test T1 vs T3: t={t13_stat:.3f}, p={t13_p:.4f}  {sig_label(t13_p)}')
print(f'Welch t-test T2 vs T3: t={t23_stat:.3f}, p={t23_p:.4f}  {sig_label(t23_p)}')

In [ ]:
# ── Let's have those boxplots: MSS trend T1→T2→T3 ──────────────────────────
# Color palette: Set1
timepoint_order  = ['T1', 'T2', 'T3']
timepoint_labels = ['T1\n(Baseline)', 'T2\n(3 months)', 'T3\n(6 months)']
box_colors       = [SET1[0], SET1[1], SET1[2]]  # red, blue, green (Set1)

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

data_by_tp = [df_mss.loc[df_mss['Timepoint'] == tp, 'MSS_NC'].values
              for tp in timepoint_order]
means_by_tp = [np.mean(d) for d in data_by_tp]

# Draw boxplots
bp = ax.boxplot(
    data_by_tp,
    positions=[1, 2, 3],
    widths=0.45,
    patch_artist=True,
    notch=False,
    medianprops=dict(color='black', linewidth=2.5),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
    flierprops=dict(marker='o', markersize=0, alpha=0),
    boxprops=dict(linewidth=1.5)
)

for patch, col in zip(bp['boxes'], box_colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.4)

# Jitter strip plot
for i, (data, col) in enumerate(zip(data_by_tp, box_colors), 1):
    jitter = np.random.normal(0, 0.07, size=len(data))
    ax.scatter(i + jitter, data, color=col, alpha=0.75, s=45, zorder=3,
               edgecolors='white', linewidths=0.5)

# Mean diamonds
for i, mean_val in enumerate(means_by_tp, 1):
    ax.plot(i, mean_val, 'D', color='black', markersize=10, zorder=5,
            markerfacecolor='white', markeredgewidth=2)

# Connect means with dashed line
ax.plot([1, 2, 3], means_by_tp, 'k--', lw=1.5, alpha=0.6, zorder=4)

# Significance brackets
def add_significance_bracket(ax, x1, x2, y, label, offset=0.02):
    top = y + offset
    ax.plot([x1, x1, x2, x2], [y, top, top, y], lw=1.5, color='black')
    ax.text((x1 + x2) / 2, top + 0.005, label,
            ha='center', va='bottom', fontsize=13, fontweight='bold')

y_max = max([d.max() for d in data_by_tp])
add_significance_bracket(ax, 1, 2, y_max + 0.04, sig_label(t12_p))
add_significance_bracket(ax, 2, 3, y_max + 0.04, sig_label(t23_p))
add_significance_bracket(ax, 1, 3, y_max + 0.10, sig_label(t13_p))

# Axes labels
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(timepoint_labels, fontsize=12)
ax.set_ylabel('Microbiota Similarity Score (MSS)\n[Probability of NC-like Profile]', fontsize=13)
ax.set_title('Gut Microbiota Similarity Score Trend\nBaseline to 6 Months',
             fontsize=14, fontweight='bold', pad=15)

# Legend
legend_patches = [
    mpatches.Patch(facecolor=SET1[0], alpha=0.5, label='T1 (Baseline)'),
    mpatches.Patch(facecolor=SET1[1], alpha=0.5, label='T2 (3 months)'),
    mpatches.Patch(facecolor=SET1[2], alpha=0.5, label='T3 (6 months)'),
    plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='white',
               markeredgecolor='black', markeredgewidth=2, markersize=9, label='Mean')
]
ax.legend(handles=legend_patches, loc='lower right', frameon=True,
          framealpha=0.9, fontsize=10)

ax.set_xlim([0.4, 3.6])
ax.set_ylim([0, y_max + 0.22])
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig('mss_trend_boxplot.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Interpretation & Selected Model Summary

best = results[best_model_name]

print('=' * 65)
print(f'  SELECTED MODEL: {best_model_name}')
print('=' * 65)
print(f'  AUC          : {best["auc"]:.4f}')
print(f'  Accuracy     : {best["accuracy"]:.4f}')
print(f'  Sensitivity  : {best["sensitivity"]:.4f}  (CRC detection rate)')
print(f'  Specificity  : {best["specificity"]:.4f}  (NC correct rejection)')
print(f'  F1 Score     : {best["f1"]:.4f}')
print(f'  Brier Score  : {brier_cal:.4f}  (calibrated MSS)')
print()
print('MSS by Timepoint:')
for tp in ['T1', 'T2', 'T3']:
    sub = df_mss[df_mss['Timepoint'] == tp]
    print(f"  {tp}: MSS_NC = {sub['MSS_NC'].mean():.4f} ± {sub['MSS_NC'].std():.4f}  "
          f"MSS_CRC = {sub['MSS_CRC'].mean():.4f} ± {sub['MSS_CRC'].std():.4f}")
print()
print('CONCLUSION:')
t3_nc_mean = df_mss[df_mss['Timepoint']=='T3']['MSS_NC'].mean()
t1_nc_mean = df_mss[df_mss['Timepoint']=='T1']['MSS_NC'].mean()
if t3_nc_mean > t1_nc_mean:
    print(f'  T3 MSS_NC ({t3_nc_mean:.4f}) > T1 MSS_NC ({t1_nc_mean:.4f})')
    print('  ✓ At 6 months, gut microbiota profiles are MORE SIMILAR to NC.')
    print('  ✓ Intervention progressively shifts microbiome toward NC-like state.')
else:
    print(f'  T3 MSS_NC ({t3_nc_mean:.4f}) ≤ T1 MSS_NC ({t1_nc_mean:.4f})')
    print('  ✗ No significant shift toward NC profile observed at 6 months.')

In [ ]:
# Save Results to Excel

with pd.ExcelWriter('gut_microbiome_ML_results.xlsx', engine='openpyxl') as writer:
    metrics_df.to_excel(writer, sheet_name='Model_Metrics', index=False)
    df_mss.to_excel(writer, sheet_name='MSS_Scores', index=False)
    wmw_df.head(20).to_excel(writer, sheet_name='Top20_WMW_Features', index=False)

print('Results saved to: gut_microbiome_ML_results.xlsx')
print('\nAll figures generated:')
print('  • metrics_table.png')
print('  • roc_curves.png')
print('  • confusion_matrices.png')
print('  • shap_beeswarm.png')
print('  • calibration_curve.png')
print('  • mss_trend_boxplot.png')

**And with that, we reach to our Goal**